# Enhanced Narrative Similarity Model

## Key Improvements:
1. **Sentence Embeddings**: Using Sentence-BERT for semantic similarity instead of string matching
2. **Temporal Ordering & Turning Points**: Identifying story progression and critical plot moments
3. **Outcome Modeling**: Explicit extraction and comparison of story endings and resolutions

## Three Core Similarity Components:
- **Abstract Theme**: Ideas, emotions, and motives
- **Course of Action**: Event sequence with temporal ordering and turning points
- **Outcomes**: Story resolutions, character end states, and final consequences

In [1]:
import os
import json
import re
from typing import List, Dict, Any, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sentence_transformers import SentenceTransformer, util
import torch


import google.generativeai as genai

print("Imports Done")

/home/saha/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports complete.


In [4]:
# Config
API_KEY = os.getenv("GEMINI_API_KEY")  
if not API_KEY:
    raise ValueError("Please set GEMINI_API_KEY environment variable")

genai.configure(api_key=API_KEY)

# Model configuration
LLM_MODEL_NAME = "gemini-2.0-flash"  # Fast and capable for event extraction
EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"  # Better quality than MiniLM

# Data paths - adjust to your directory structure
DEV_PATH = "../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl"
SAMPLE_PATH = "../Data/SemEval2026-Task_4-sample-v1/sample_track_a.jsonl"

print(f"LLM Model: {LLM_MODEL_NAME}")
print(f"Embedding Model: {EMBEDDING_MODEL_NAME}")
print(f"Data path: {DEV_PATH}")

LLM Model: gemini-2.0-flash
Embedding Model: all-mpnet-base-v2
Data path: ../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl


In [6]:
# Initialize models
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

llm_model = genai.GenerativeModel(LLM_MODEL_NAME)
print("Models read")

Loading sentence embedding model...
Embedding dimension: 768

Initializing LLM for event extraction...
Models ready!


In [7]:
@dataclass
class Event:
    """Represents a single narrative event"""
    description: str
    position: int  # Position in story (0-based index)
    is_turning_point: bool = False
    temporal_marker: str = ""  # e.g., "before", "after", "meanwhile"
    importance_score: float = 0.5  # 0-1, how central this event is
    
@dataclass
class Outcome:
    """Represents story outcomes and resolutions"""
    main_resolution: str = ""
    character_states: List[str] = field(default_factory=list)
    consequences: List[str] = field(default_factory=list)
    
@dataclass
class NarrativeRepresentation:
    """Complete narrative representation"""
    text: str
    theme: str = ""  # Abstract theme/ideas
    events: List[Event] = field(default_factory=list)
    outcome: Outcome = field(default_factory=Outcome)
    
    def get_event_descriptions(self) -> List[str]:
        return [e.description for e in self.events]
    
    def get_turning_points(self) -> List[Event]:
        return [e for e in self.events if e.is_turning_point]

print("Data structures defined.")

Data structures defined.


In [8]:
## 1. Semantic Similarity (Sentence Embeddings)

In [9]:
def compute_semantic_similarity(text1: str, text2: str) -> float:
    """
    Compute semantic similarity using sentence embeddings.
    Returns cosine similarity between 0 and 1.
    """
    if not text1 or not text2:
        return 0.0
    
    if not text1.strip() or not text2.strip():
        return 0.0
    
    # Encode texts to embeddings
    emb1 = embedding_model.encode(text1, convert_to_tensor=True)
    emb2 = embedding_model.encode(text2, convert_to_tensor=True)
    
    # Compute cosine similarity
    similarity = util.cos_sim(emb1, emb2)
    return float(similarity.item())


def compute_batch_similarities(texts1: List[str], texts2: List[str]) -> np.ndarray:
    """
    Compute pairwise similarities between two lists of texts.
    Returns matrix of shape (len(texts1), len(texts2)).
    """
    if not texts1 or not texts2:
        return np.zeros((len(texts1), len(texts2)))
    
    # Batch encode for efficiency
    embs1 = embedding_model.encode(texts1, convert_to_tensor=True)
    embs2 = embedding_model.encode(texts2, convert_to_tensor=True)
    
    # Compute all pairwise similarities
    similarities = util.cos_sim(embs1, embs2)
    return similarities.cpu().numpy()


# Test semantic similarity
test_cases = [
    ("The hero defeated the villain", "The protagonist vanquished the antagonist"),
    ("A lost item was found", "An object was discovered"),
    ("The sun is bright", "The cat ate fish")
]

print("Testing semantic similarity:")
for text1, text2 in test_cases:
    sim = compute_semantic_similarity(text1, text2)
    print(f"  '{text1}' <-> '{text2}': {sim:.3f}")

print("\nSemantic similarity functions")

Testing semantic similarity:
  'The hero defeated the villain' <-> 'The protagonist vanquished the antagonist': 0.848
  'A lost item was found' <-> 'An object was discovered': 0.613
  'The sun is bright' <-> 'The cat ate fish': 0.046

Semantic similarity functions ready!


In [10]:
## 2. Temporal Ordering & Turning Point Detection

In [11]:
def extract_events_with_temporal_info(text: str, use_llm: bool = True) -> List[Event]:
    """
    Extract events with temporal ordering and identify turning points.
    Uses LLM for intelligent extraction.
    """
    if use_llm:
        prompt = f"""Analyze this narrative and extract key events. For each event:
1. Describe it briefly (1 sentence)
2. Mark if it's a TURNING POINT (major plot shift)
3. Note any temporal marker (before/after/meanwhile/then/etc.)
4. Rate importance (0.0-1.0)

Focus on extracting 5-8 central events that drive the story forward.

NARRATIVE:
{text}

Return ONLY a JSON array like this:
[
  {{
    "description": "event description",
    "is_turning_point": true/false,
    "temporal_marker": "then/after/etc.",
    "importance_score": 0.8
  }},
  ...
]
"""
        
        try:
            response = llm_model.generate_content(prompt)
            # Extract JSON from response
            response_text = response.text.strip()
            # Remove markdown code blocks if present
            response_text = re.sub(r'^```json\s*', '', response_text)
            response_text = re.sub(r'\s*```$', '', response_text)
            
            events_data = json.loads(response_text)
            
            events = []
            for idx, event_dict in enumerate(events_data):
                events.append(Event(
                    description=event_dict.get('description', ''),
                    position=idx,
                    is_turning_point=event_dict.get('is_turning_point', False),
                    temporal_marker=event_dict.get('temporal_marker', ''),
                    importance_score=event_dict.get('importance_score', 0.5)
                ))
            
            return events
            
        except Exception as e:
            print(f"LLM extraction failed: {e}")
            # Fallback to simple extraction
            return _simple_event_extraction(text)
    else:
        return _simple_event_extraction(text)


def _simple_event_extraction(text: str) -> List[Event]:
    """
    Fallback: Simple sentence-based event extraction.
    """
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 20]
    events = []
    
    # Simple heuristics for turning points
    turning_point_keywords = ['however', 'but', 'suddenly', 'then', 'finally', 
                              'discovered', 'realized', 'decided', 'changed']
    
    for idx, sentence in enumerate(sentences):
        is_turning = any(kw in sentence.lower() for kw in turning_point_keywords)
        importance = 0.7 if is_turning else 0.5
        
        events.append(Event(
            description=sentence,
            position=idx,
            is_turning_point=is_turning,
            temporal_marker='',
            importance_score=importance
        ))
    
    return events


def compute_temporal_order_similarity(events1: List[Event], events2: List[Event]) -> float:
    """
    Compare temporal ordering of events between two stories.
    Uses semantic similarity to match events, then checks if relative order is preserved.
    """
    if not events1 or not events2:
        return 0.0
    
    # Get event descriptions
    desc1 = [e.description for e in events1]
    desc2 = [e.description for e in events2]
    
    # Compute similarity matrix
    sim_matrix = compute_batch_similarities(desc1, desc2)
    
    # Find best matches (threshold for considering a match)
    threshold = 0.5
    matches = []
    
    for i in range(len(events1)):
        for j in range(len(events2)):
            if sim_matrix[i, j] > threshold:
                matches.append((i, j, sim_matrix[i, j]))
    
    if not matches:
        return 0.0
    
    # Check how many match pairs preserve relative order
    order_preserved = 0
    total_comparisons = 0
    
    for idx1, (i1, j1, _) in enumerate(matches):
        for idx2, (i2, j2, _) in enumerate(matches[idx1+1:], idx1+1):
            total_comparisons += 1
            # If relative order is same in both stories
            if (i1 < i2 and j1 < j2) or (i1 > i2 and j1 > j2):
                order_preserved += 1
    
    if total_comparisons == 0:
        return 0.5  # Only one match, can't determine order
    
    return order_preserved / total_comparisons


def compute_turning_point_similarity(events1: List[Event], events2: List[Event]) -> float:
    """
    Compare turning points between two stories.
    Turning points are critical plot moments and should be similar.
    """
    tp1 = [e.description for e in events1 if e.is_turning_point]
    tp2 = [e.description for e in events2 if e.is_turning_point]
    
    if not tp1 or not tp2:
        # If neither has turning points, they're similar in that regard
        if not tp1 and not tp2:
            return 0.5
        # If one has turning points and other doesn't, less similar
        return 0.2
    
    # Compute pairwise similarities between turning points
    sim_matrix = compute_batch_similarities(tp1, tp2)
    
    # Use Hungarian algorithm for optimal matching
    # Convert to cost matrix (1 - similarity)
    cost_matrix = 1 - sim_matrix
    
    # Pad matrix if different sizes
    max_size = max(len(tp1), len(tp2))
    padded_cost = np.ones((max_size, max_size))
    padded_cost[:cost_matrix.shape[0], :cost_matrix.shape[1]] = cost_matrix
    
    # Find optimal assignment
    row_ind, col_ind = linear_sum_assignment(padded_cost)
    
    # Calculate average similarity of matched turning points
    valid_matches = [(i, j) for i, j in zip(row_ind, col_ind) 
                    if i < len(tp1) and j < len(tp2)]
    
    if not valid_matches:
        return 0.0
    
    total_sim = sum(sim_matrix[i, j] for i, j in valid_matches)
    return total_sim / len(valid_matches)


print("Temporal ordering and turning point functions defined.")

Temporal ordering and turning point functions defined.


In [12]:
## 3. Outcome Modeling

In [13]:
def extract_outcomes(text: str, events: List[Event], use_llm: bool = True) -> Outcome:
    """
    Extract story outcomes and resolutions.
    Focuses on the ending and final consequences.
    """
    if use_llm:
        prompt = f"""Analyze the ENDING and OUTCOMES of this narrative.

Extract:
1. Main resolution: How the central conflict/story resolves
2. Character end states: Final status/situation of main characters (list)
3. Consequences: Long-term results or implications (list)

Focus on the ENDING, not the middle of the story.

NARRATIVE:
{text}

Return ONLY valid JSON:
{{
  "main_resolution": "brief description of how story ends",
  "character_states": ["character 1 ends up...", "character 2 is..."],
  "consequences": ["consequence 1", "consequence 2"]
}}
"""
        
        try:
            response = llm_model.generate_content(prompt)
            response_text = response.text.strip()
            # Remove markdown code blocks
            response_text = re.sub(r'^```json\s*', '', response_text)
            response_text = re.sub(r'\s*```$', '', response_text)
            
            outcome_data = json.loads(response_text)
            
            return Outcome(
                main_resolution=outcome_data.get('main_resolution', ''),
                character_states=outcome_data.get('character_states', []),
                consequences=outcome_data.get('consequences', [])
            )
            
        except Exception as e:
            print(f"LLM outcome extraction failed: {e}")
            return _simple_outcome_extraction(text, events)
    else:
        return _simple_outcome_extraction(text, events)


def _simple_outcome_extraction(text: str, events: List[Event]) -> Outcome:
    """
    Fallback: Use last few sentences as outcome.
    """
    sentences = [s.strip() for s in text.split('.') if s.strip()]
    
    # Take last 20% of text as outcome
    outcome_start = int(len(sentences) * 0.8)
    outcome_sentences = sentences[outcome_start:]
    
    return Outcome(
        main_resolution=' '.join(outcome_sentences) if outcome_sentences else text[-200:],
        character_states=[],
        consequences=[]
    )


def compute_outcome_similarity(outcome1: Outcome, outcome2: Outcome) -> float:
    """
    Compare outcomes between two stories.
    Considers resolution, character states, and consequences.
    """
    similarities = []
    
    # Compare main resolutions
    if outcome1.main_resolution and outcome2.main_resolution:
        res_sim = compute_semantic_similarity(
            outcome1.main_resolution, 
            outcome2.main_resolution
        )
        similarities.append(res_sim)
    
    # Compare character states
    if outcome1.character_states and outcome2.character_states:
        char_sims = compute_batch_similarities(
            outcome1.character_states, 
            outcome2.character_states
        )
        # Take best matches
        if char_sims.size > 0:
            similarities.append(np.max(char_sims, axis=1).mean())
    
    # Compare consequences
    if outcome1.consequences and outcome2.consequences:
        cons_sims = compute_batch_similarities(
            outcome1.consequences, 
            outcome2.consequences
        )
        if cons_sims.size > 0:
            similarities.append(np.max(cons_sims, axis=1).mean())
    
    if not similarities:
        return 0.0
    
    # Weight main resolution more heavily
    if len(similarities) == 1:
        return similarities[0]
    else:
        weights = [0.5] + [0.5 / (len(similarities) - 1)] * (len(similarities) - 1)
        return sum(s * w for s, w in zip(similarities, weights))


print("Outcome modeling functions defined.")

Outcome modeling functions defined.


In [14]:
## Theme Extraction

In [15]:
def extract_theme(text: str, use_llm: bool = True) -> str:
    """
    Extract abstract theme/ideas from the narrative.
    """
    if use_llm:
        prompt = f"""What is the abstract THEME of this narrative? 

Focus on:
- Core ideas and concepts (e.g., redemption, revenge, love, sacrifice)
- Underlying motives and emotional journeys
- Universal patterns or moral lessons

NARRATIVE:
{text}

Respond with a single paragraph (2-3 sentences) describing the theme.
"""
        
        try:
            response = llm_model.generate_content(prompt)
            return response.text.strip()
        except Exception as e:
            print(f"Theme extraction failed: {e}")
            # Fallback: use first few sentences
            return text[:300]
    else:
        return text[:300]


print("Theme extraction function defined.")

Theme extraction function defined.


In [16]:
## Complete Narrative Analysis Pipeline

In [17]:
def analyze_narrative(text: str, use_llm: bool = True) -> NarrativeRepresentation:
    """
    Complete pipeline: extract all narrative components.
    """
    # Extract theme
    theme = extract_theme(text, use_llm=use_llm)
    
    # Extract events with temporal info
    events = extract_events_with_temporal_info(text, use_llm=use_llm)
    
    # Extract outcomes
    outcome = extract_outcomes(text, events, use_llm=use_llm)
    
    return NarrativeRepresentation(
        text=text,
        theme=theme,
        events=events,
        outcome=outcome
    )


def compare_narratives(
    narrative1: NarrativeRepresentation, 
    narrative2: NarrativeRepresentation,
    weights: Optional[Dict[str, float]] = None
) -> Dict[str, float]:
    """
    Compare two narratives across all three dimensions.
    
    Returns:
        Dictionary with component scores and overall similarity
    """
    if weights is None:
        weights = {
            'theme': 0.25,
            'events': 0.35,
            'temporal_order': 0.15,
            'turning_points': 0.10,
            'outcome': 0.15
        }
    
    # 1. Theme similarity
    theme_sim = compute_semantic_similarity(narrative1.theme, narrative2.theme)
    
    # 2. Event similarity (average of all event pairs)
    event_desc1 = narrative1.get_event_descriptions()
    event_desc2 = narrative2.get_event_descriptions()
    
    if event_desc1 and event_desc2:
        event_sims = compute_batch_similarities(event_desc1, event_desc2)
        # Use Hungarian algorithm for optimal matching
        cost_matrix = 1 - event_sims
        max_size = max(len(event_desc1), len(event_desc2))
        padded_cost = np.ones((max_size, max_size))
        padded_cost[:cost_matrix.shape[0], :cost_matrix.shape[1]] = cost_matrix
        
        row_ind, col_ind = linear_sum_assignment(padded_cost)
        valid_matches = [(i, j) for i, j in zip(row_ind, col_ind) 
                        if i < len(event_desc1) and j < len(event_desc2)]
        
        if valid_matches:
            event_sim = sum(event_sims[i, j] for i, j in valid_matches) / len(valid_matches)
        else:
            event_sim = 0.0
    else:
        event_sim = 0.0
    
    # 3. Temporal order similarity
    order_sim = compute_temporal_order_similarity(narrative1.events, narrative2.events)
    
    # 4. Turning point similarity
    tp_sim = compute_turning_point_similarity(narrative1.events, narrative2.events)
    
    # 5. Outcome similarity
    outcome_sim = compute_outcome_similarity(narrative1.outcome, narrative2.outcome)
    
    # Compute weighted overall similarity
    overall = (
        weights['theme'] * theme_sim +
        weights['events'] * event_sim +
        weights['temporal_order'] * order_sim +
        weights['turning_points'] * tp_sim +
        weights['outcome'] * outcome_sim
    )
    
    return {
        'theme_similarity': theme_sim,
        'event_similarity': event_sim,
        'temporal_order_similarity': order_sim,
        'turning_point_similarity': tp_sim,
        'outcome_similarity': outcome_sim,
        'overall_similarity': overall
    }


print("Complete analysis pipeline ready!")

Complete analysis pipeline ready!


In [18]:
## Testing on Sample Data

In [19]:
# Load data
try:
    df_dev = pd.read_json(DEV_PATH, lines=True)
    print(f"Loaded {len(df_dev)} examples from dev set")
    print(f"Columns: {df_dev.columns.tolist()}")
    print(f"\nFirst example:")
    print(df_dev.iloc[0])
except FileNotFoundError:
    print(f"Dev file not found at {DEV_PATH}")
    print("Please update the DEV_PATH variable with the correct path.")
    df_dev = None

Loaded 200 examples from dev set
Columns: ['anchor_text', 'text_a', 'text_b', 'text_a_is_closer']

First example:
anchor_text         The book follows an international organization...
text_a              The old grandmother Tina arrives in town to at...
text_b              The nano-plague that poisoned Earth's water su...
text_a_is_closer                                                False
Name: 0, dtype: object


In [22]:
# Test on a single example
if df_dev is not None and len(df_dev) > 0:
    print("Analyzing first example...\n")
    
    row = df_dev.iloc[0]
    anchor_text = row['anchor_text']
    text_a = row['text_a']
    text_b = row['text_b']
    
    print("=" * 80)
    print("ANCHOR:", anchor_text[:200], "...")
    print("\nTEXT A:", text_a[:200], "...")
    print("\nTEXT B:", text_b[:200], "...")
    print("=" * 80)
    
    # Analyze all three narratives
    print("\n Analyzing anchor...")
    anchor_rep = analyze_narrative(anchor_text, use_llm=True)
    
    print("\n Analyzing text A...")
    text_a_rep = analyze_narrative(text_a, use_llm=True)
    
    print("\n Analyzing text B...")
    text_b_rep = analyze_narrative(text_b, use_llm=True)
    
    print("\n" + "=" * 80)
    print("ANALYSIS RESULTS")
    print("=" * 80)
    
    # Display extracted components
    print("\n ANCHOR ANALYSIS:")
    print(f"Theme: {anchor_rep.theme}")
    print(f"\nEvents ({len(anchor_rep.events)}):")
    for i, event in enumerate(anchor_rep.events, 1):
        tp_marker = " [TURNING POINT]" if event.is_turning_point else ""
        print(f"  {i}. {event.description[:100]}...{tp_marker}")
    print(f"\nOutcome: {anchor_rep.outcome.main_resolution[:200]}...")
    
    # Compare
    print("\n" + "=" * 80)
    print("SIMILARITY COMPARISON")
    print("=" * 80)
    
    comparison_a = compare_narratives(anchor_rep, text_a_rep)
    comparison_b = compare_narratives(anchor_rep, text_b_rep)
    
    print("\n Anchor vs Text A:")
    for key, value in comparison_a.items():
        print(f"  {key}: {value:.3f}")
    
    print("\n Anchor vs Text B:")
    for key, value in comparison_b.items():
        print(f"  {key}: {value:.3f}")
    
    # Prediction
    predicted = 'A' if comparison_a['overall_similarity'] > comparison_b['overall_similarity'] else 'B'
    actual = 'A' if row['text_a_is_closer'] else 'B'
    correct = "✓" if predicted == actual else "✗"
    
    print(f"\n{correct} PREDICTION: Text {predicted} is more similar")
    print(f"   ACTUAL: Text {actual} is more similar")
    print(f"   Margin: {abs(comparison_a['overall_similarity'] - comparison_b['overall_similarity']):.3f}")

Analyzing first example...

ANCHOR: The book follows an international organization named the Ministry for the Future in its mission to act as an advocate for the world's future generations of citizens as if their rights were as valid as ...

TEXT A: The old grandmother Tina arrives in town to attend the wedding of his nephew Alberto with his girlfriend Ileana.
Upon arrival she discovers that she has been stolen of a medallion that her late husban ...

TEXT B: The nano-plague that poisoned Earth's water supply has reached its 60-year critical mass. The Unlight enemy forced the first exodus to the moon where the outlawed banished population was supposed to d ...

 Analyzing anchor...

 Analyzing text A...

 Analyzing text B...

ANALYSIS RESULTS

 ANCHOR ANALYSIS:
Theme: The narrative's central theme revolves around humanity's urgent need for proactive, systemic change to safeguard the future from the devastating consequences of climate change. It explores the potential for global coopera

In [23]:
def evaluate_model(
    df: pd.DataFrame, 
    max_samples: Optional[int] = None,
    weights: Optional[Dict[str, float]] = None,
    use_llm: bool = True,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Evaluate the model on a dataset.
    """
    if max_samples:
        df = df.head(max_samples)
    
    correct = 0
    total = len(df)
    results = []
    
    for idx, row in df.iterrows():
        try:
            # Analyze narratives
            anchor_rep = analyze_narrative(row['anchor_text'], use_llm=use_llm)
            text_a_rep = analyze_narrative(row['text_a'], use_llm=use_llm)
            text_b_rep = analyze_narrative(row['text_b'], use_llm=use_llm)
            
            # Compare
            comp_a = compare_narratives(anchor_rep, text_a_rep, weights=weights)
            comp_b = compare_narratives(anchor_rep, text_b_rep, weights=weights)
            
            # Predict
            predicted = 'A' if comp_a['overall_similarity'] > comp_b['overall_similarity'] else 'B'
            actual = 'A' if row['text_a_is_closer'] else 'B'
            is_correct = (predicted == actual)
            
            if is_correct:
                correct += 1
            
            result = {
                'index': idx,
                'predicted': predicted,
                'actual': actual,
                'correct': is_correct,
                'sim_a': comp_a['overall_similarity'],
                'sim_b': comp_b['overall_similarity'],
                'margin': abs(comp_a['overall_similarity'] - comp_b['overall_similarity']),
                'components_a': comp_a,
                'components_b': comp_b
            }
            results.append(result)
            
            if verbose:
                status = "✓" if is_correct else "✗"
                print(f"{idx+1}/{total}: {status} pred={predicted}, actual={actual}, "
                      f"margin={result['margin']:.3f}")
        
        except Exception as e:
            print(f"Error processing example {idx}: {e}")
            continue
    
    accuracy = correct / total if total > 0 else 0
    
    print("\n" + "=" * 80)
    print(f"ACCURACY: {accuracy:.4f} ({correct}/{total} correct)")
    print("=" * 80)
    
    return {
        'accuracy': accuracy,
        'correct': correct,
        'total': total,
        'results': results
    }


print("Evaluation function ready!")

Evaluation function ready!


In [24]:
# Run evaluation on a subset first
if df_dev is not None:
    print("Evaluating on 10 samples...\n")
    
    eval_results = evaluate_model(
        df_dev,
        max_samples=10,
        weights=None,  # Use default weights
        use_llm=True,
        verbose=True
    )
    
    # Analyze results
    print("\n Component Score Analysis:")
    
    errors = [r for r in eval_results['results'] if not r['correct']]
    corrects = [r for r in eval_results['results'] if r['correct']]
    
    if errors:
        print(f"\nErrors ({len(errors)}):")
        print(f"  Average margin: {np.mean([e['margin'] for e in errors]):.3f}")
    
    if corrects:
        print(f"\nCorrect ({len(corrects)}):")
        print(f"  Average margin: {np.mean([c['margin'] for c in corrects]):.3f}")

Evaluating on 10 samples...

1/10: ✓ pred=B, actual=B, margin=0.077
2/10: ✗ pred=B, actual=A, margin=0.034
3/10: ✗ pred=A, actual=B, margin=0.014
4/10: ✗ pred=A, actual=B, margin=0.007
5/10: ✓ pred=B, actual=B, margin=0.141
6/10: ✗ pred=A, actual=B, margin=0.093
7/10: ✓ pred=B, actual=B, margin=0.025
8/10: ✓ pred=B, actual=B, margin=0.076
9/10: ✗ pred=B, actual=A, margin=0.017
10/10: ✓ pred=A, actual=A, margin=0.014

ACCURACY: 0.5000 (5/10 correct)

 Component Score Analysis:

Errors (5):
  Average margin: 0.033

Correct (5):
  Average margin: 0.067
